# `effectful_mcmc` quickstart

A walkthrough of the bridge over `numpyro.infer.NUTS` for `effectful` models. Three demos:

1. **Bayesian linear regression** on a synthetic dataset — basic `sample`/`MCMC`/`NUTS` usage.
2. **Eight schools** (Rubin, 1981) — a real hierarchical-Bayes benchmark, written in the per-element idiom (one explicit `sample` per school).
3. **Counterfactual via `Intervene`** — what the bridge buys you over plain NumPyro: handlers compose with the inference layer, so you can intervene on a named site mid-program and re-run inference under the intervention.

Distributions come from `effectful.handlers.numpyro` (term-aware: they accept both eager arrays and effectful terms in the same call). The primitives `sample`, `factor`, `param`, `deterministic` are `@defop`s — interceptable by any effectful handler stack.

Run locally:

```bash
pip install -e .
jupyter notebook docs/quickstart.ipynb
```

In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr

from effectful.handlers.numpyro import Normal, HalfNormal
from effectful.ops.semantics import handler

from effectful_mcmc import sample, MCMC, NUTS, Intervene

jax.config.update("jax_platform_name", "cpu")  # demo is small enough that CPU is fastest

/Users/datnguyen/HMC_NUTS_Effectful/effectful_mcmc/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Bayesian linear regression on synthetic data

Generate a small dataset with known parameters: `y = α + β·x + noise`, with `α=2.0`, `β=-1.5`, `noise ~ Normal(0, 0.5)`. Then infer the posterior over `α, β, σ` and check we recover the truth.

The model uses `effectful.handlers.numpyro`'s `Normal`/`HalfNormal` so the priors over `α`, `β`, `σ` produce term-aware distributions that can plug back into the likelihood `Normal(α + β·x, σ)` with no glue code.

In [2]:
# Synthetic regression dataset
true_alpha, true_beta, true_sigma = 2.0, -1.5, 0.5
N = 50

key_data = jr.PRNGKey(0)
k_x, k_noise = jr.split(key_data)
x_data = jr.uniform(k_x, (N,), minval=-2.0, maxval=2.0)
y_data = true_alpha + true_beta * x_data + true_sigma * jr.normal(k_noise, (N,))

print(f"Generated {N} points with α={true_alpha}, β={true_beta}, σ={true_sigma}")

Generated 50 points with α=2.0, β=-1.5, σ=0.5


In [3]:
def linear_regression(x, y):
    alpha = sample(Normal(0.0, 10.0), name="alpha")
    beta  = sample(Normal(0.0, 10.0), name="beta")
    sigma = sample(HalfNormal(2.0),    name="sigma")
    sample(Normal(alpha + beta * x, sigma), obs=y, name="obs")
    return alpha, beta, sigma

mcmc = MCMC(NUTS(linear_regression), num_warmup=500, num_samples=1000, progress_bar=False)
mcmc.run(jr.PRNGKey(1), x_data, y_data)

samples = mcmc.get_samples_by_name()
for name, true in [("alpha", true_alpha), ("beta", true_beta), ("sigma", true_sigma)]:
    post = samples[name]
    print(f"{name}: posterior mean = {float(post.mean()):+.3f}  std = {float(post.std()):.3f}  (truth = {true:+.2f})")

alpha: posterior mean = +1.771  std = 0.083  (truth = +2.00)
beta: posterior mean = -1.481  std = 0.065  (truth = -1.50)
sigma: posterior mean = +0.570  std = 0.059  (truth = +0.50)


Posteriors recover the truth within ~1 posterior std. `mcmc.print_summary()` works the same as `numpyro.infer.MCMC.print_summary()` — the bridge forwards post-run attribute access via `__getattr__`:

In [4]:
mcmc.print_summary()


                mean       std    median      5.0%     95.0%     n_eff     r_hat
     alpha      1.77      0.08      1.77      1.64      1.91   1080.79      1.00
      beta     -1.48      0.06     -1.48     -1.59     -1.38    841.94      1.00
     sigma      0.57      0.06      0.56      0.48      0.67    862.90      1.00

Number of divergences: 0


### Operation-handle access (no string names)

The string-name pattern (`name="alpha"`) is one of two access patterns. The other is **Operation-handle**: the model returns the per-site terms, and you index `mcmc.get_samples()` (no `_by_name`) with the underlying `Operation` via `.op`.

This makes posteriors refactor-safe — there's no string to keep in sync between the model and downstream code:

In [5]:
alpha_t, beta_t, sigma_t = mcmc.model_return_value
by_op = mcmc.get_samples()
print(f"alpha posterior mean (by Operation): {float(by_op[alpha_t.op].mean()):+.3f}")
print(f"beta  posterior mean (by Operation): {float(by_op[beta_t.op].mean()):+.3f}")
print(f"sigma posterior mean (by Operation): {float(by_op[sigma_t.op].mean()):+.3f}")

alpha posterior mean (by Operation): +1.771
beta  posterior mean (by Operation): -1.481
sigma posterior mean (by Operation): +0.570


## 2. Eight schools (Rubin, 1981) — hierarchical Bayes

Classical hierarchical-Bayes benchmark: eight schools each report an estimated treatment effect `y_j` with reported standard error `σ_j`. The hierarchical model partially pools the school-level effects toward a population mean `μ` with population scale `τ`:

$$
\mu \sim \mathcal{N}(0, 10) \\
\tau \sim \mathrm{HalfNormal}(10) \\
\theta_j \sim \mathcal{N}(\mu, \tau) \\
y_j \sim \mathcal{N}(\theta_j, \sigma_j)
$$

Written in the per-element idiom — one explicit `sample` per school — which the bridge supports today as a hierarchical-model story (no `plate` machinery needed).

In [6]:
# Rubin (1981) data
y_obs    = jnp.array([28.0,  8.0, -3.0,  7.0, -1.0,  1.0, 18.0, 12.0])
sigma_obs = jnp.array([15.0, 10.0, 16.0, 11.0,  9.0, 11.0, 10.0, 18.0])
J = len(y_obs)

def eight_schools(y, sigma):
    mu  = sample(Normal(0.0, 10.0), name="mu")
    tau = sample(HalfNormal(10.0),   name="tau")
    for j in range(J):
        theta_j = sample(Normal(mu, tau), name=f"theta_{j}")
        sample(Normal(theta_j, sigma[j]), obs=y[j], name=f"y_{j}")

mcmc8 = MCMC(NUTS(eight_schools), num_warmup=1000, num_samples=2000, progress_bar=False)
mcmc8.run(jr.PRNGKey(2), y_obs, sigma_obs)
samples8 = mcmc8.get_samples_by_name()

print(f"μ  posterior: mean = {float(samples8['mu'].mean()):+.2f}  std = {float(samples8['mu'].std()):.2f}")
print(f"τ  posterior: mean = {float(samples8['tau'].mean()):+.2f}  std = {float(samples8['tau'].std()):.2f}")
print("per-school θ posteriors:")
for j in range(J):
    th = samples8[f"theta_{j}"]
    print(f"  school {j}: mean = {float(th.mean()):+6.2f}  std = {float(th.std()):.2f}  (raw y_{j} = {float(y_obs[j]):+.1f})")

μ  posterior: mean = +5.92  std = 3.03
τ  posterior: mean = +2.33  std = 3.15
per-school θ posteriors:
  school 0: mean =  +6.99  std = 4.66  (raw y_0 = +28.0)
  school 1: mean =  +6.09  std = 3.97  (raw y_1 = +8.0)
  school 2: mean =  +5.57  std = 4.41  (raw y_2 = -3.0)
  school 3: mean =  +6.01  std = 4.03  (raw y_3 = +7.0)
  school 4: mean =  +5.28  std = 4.03  (raw y_4 = -1.0)
  school 5: mean =  +5.45  std = 4.04  (raw y_5 = +1.0)
  school 6: mean =  +6.89  std = 4.11  (raw y_6 = +18.0)
  school 7: mean =  +6.09  std = 4.57  (raw y_7 = +12.0)


Note the **shrinkage**: per-school posterior means are pulled toward `μ ≈ 7` from the raw observations. School 0 (`y_0 = +28`) is the most dramatic — the hierarchical prior pulls its `θ_0` down because the population-scale `τ` is small enough that 28 looks like a noise excursion.

## 3. Counterfactual via `Intervene` — what the bridge buys you

Plain NumPyro has `numpyro.handlers.substitute` which can fix a site to a value, but it's a NumPyro-specific mechanism that doesn't compose with other effectful handlers, and it doesn't have a clean story for "intervene then re-run inference under the intervention".

Here's the bridge's story:

- `Intervene(name, value)` is an effectful handler. It intercepts the `sample` defop and overrides the named site.
- Wrap the model in `with handler(Intervene(...))` and pass the wrapped model to `MCMC(NUTS(...))`. The intervention survives all the way through compile-time tracing and per-step potential evaluation.
- The intervened site is **removed** from the posterior — the user already passed the value in; they don't need MCMC to round-trip it.
- The remaining sites' posteriors **tighten** because they now condition on the intervention as if it were observed.

We demonstrate by asking: *what would the posterior over `α` and `β` look like in our linear regression if we knew the noise level was exactly `σ = 0.5`?*

In [7]:
def regression_with_sigma_fixed(x, y):
    with handler(Intervene("sigma", jnp.array(0.5))):
        return linear_regression(x, y)

mcmc_fixed = MCMC(NUTS(regression_with_sigma_fixed),
                  num_warmup=500, num_samples=1000, progress_bar=False)
mcmc_fixed.run(jr.PRNGKey(3), x_data, y_data)
samples_fixed = mcmc_fixed.get_samples_by_name()

print("baseline (σ inferred) vs. intervened (σ = 0.5 fixed):\n")
print(f"{'param':>10s}  {'baseline mean ± std':>22s}  {'intervened mean ± std':>23s}  {'std ratio':>10s}")
print("-" * 75)
for name in ("alpha", "beta"):
    b_mean, b_std = float(samples[name].mean()),       float(samples[name].std())
    i_mean, i_std = float(samples_fixed[name].mean()), float(samples_fixed[name].std())
    print(f"{name:>10s}  {b_mean:+.3f} ± {b_std:.3f}        {i_mean:+.3f} ± {i_std:.3f}         {i_std/b_std:.3f}")

print(f"\nintervened-σ samples keys: {sorted(samples_fixed.keys())}")
print(f"  ('sigma' is absent — it's the intervened site, removed from posterior)")

baseline (σ inferred) vs. intervened (σ = 0.5 fixed):

     param     baseline mean ± std    intervened mean ± std   std ratio
---------------------------------------------------------------------------
     alpha  +1.771 ± 0.083        +1.769 ± 0.070         0.849
      beta  -1.481 ± 0.065        -1.477 ± 0.057         0.887

intervened-σ samples keys: ['alpha', 'beta']
  ('sigma' is absent — it's the intervened site, removed from posterior)


Posteriors over `α` and `β` are essentially unchanged in mean (the data already identifies them well) but slightly tighter when `σ` is held fixed, because the inferred model no longer has to absorb noise-level uncertainty into the slope/intercept posterior.

The point of this last demo isn't the numerical effect — it's that `Intervene` is **just a handler**. It's a tiny class (~10 LOC, in `effectful_mcmc/__init__.py`) that intercepts the abstract `sample` defop. The same handler works under `MCMC`, under prior simulation (no MCMC at all), under SVI, and under any custom inference scheme that routes through `sample`. There's no MCMC-specific intervention API to learn — write the handler once, use it everywhere.

Custom handlers (conditioning, masking, ablations, projections, ...) follow the same pattern: subclass `ObjectInterpretation`, decorate methods with `@implements(sample)` / `@implements(factor)` / etc., and `with handler(MyHandler())` to stack them around the model.

## Where to go next

- `tests/test_composition.py` — the differentiating tests: in-trace intervention, `fwd()` correctness, handler-nesting composition, hierarchical eight-schools, symbolic-indexing under MCMC.
- `tests/test_distributions.py` — every distribution in `effectful.handlers.numpyro` certified through NUTS (modulo discrete-skip and a structural `Delta` exception).
- `docs/biject_to_compatibility.md` — `biject_to` spike output: which distributions need workarounds for transform-based inference.